<a href="https://colab.research.google.com/github/surue-s/BreakFixPro/blob/main/4k_Video_Upscaler_Colab_(Real_ESRGAN).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# @title setup realesrgan environment {display-mode: "code"}
import os, torch

assert torch.cuda.is_available(), "gpu not detected, change runtime to gpu"

!rm -rf Real-ESRGAN
!git clone https://github.com/xinntao/Real-ESRGAN.git
%cd Real-ESRGAN

!pip install -q basicsr facexlib gfpgan ffmpeg-python ipywidgets
!pip install -q -r requirements.txt
!pip install -e .

# fix torchvision import for basicsr
import fileinput
file_path = "/usr/local/lib/python3.12/dist-packages/basicsr/data/degradations.py"

for line in fileinput.input(file_path, inplace=True):
    if "functional_tensor" in line:
        print("from torchvision.transforms.functional import rgb_to_grayscale")
    else:
        print(line, end="")

In [ ]:
# @title video upscaler settings {display-mode: "form"}
video_path = "/content/video.mp4" # @param {type:"string"}
output_dir = "/content/output"    # @param {type:"string"}
resolution = "2k (2560 x 1440)"   # @param ["FHD (1920 x 1080)", "2k (2560 x 1440)", "4k (3840 x 2160)","2 x original", "3 x original", "4 x original"]
model = "RealESRGAN_x4plus"       # @param ["RealESRGAN_x4plus", "RealESRGAN_x4plus_anime_6B", "realesr-animevideov3"]
mount_drive = False                # @param {type:"boolean"}

# @title run upscaling {display-mode: "code"}
import os, cv2, subprocess

assert os.path.exists(video_path), "video file does not exist"
os.makedirs(output_dir, exist_ok=True)

cap = cv2.VideoCapture(video_path)
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
aspect = w / h

# get final width and height
final_width = None
final_height = None

match resolution:
    case "FHD (1920 x 1080)":
        final_width, final_height = 1920, 1080
    case "2k (2560 x 1440)":
        final_width, final_height = 2560, 1440
    case "4k (3840 x 2160)":
        final_width, final_height = 3840, 2160
    case "2 x original":
        final_width, final_height = 2*w, 2*h
    case "3 x original":
        final_width, final_height = 3*w, 3*h
    case "4 x original":
        final_width, final_height = 4*w, 4*h

if aspect == 1.0 and "original" not in resolution:
    final_height = final_width
elif aspect < 1.0 and "original" not in resolution:
    final_width, final_height = final_height, final_width

scale_factor = max(final_width/w, final_height/h)
while int(w*scale_factor)%2 != 0 or int(h*scale_factor)%2 != 0:
    scale_factor += 0.01

print(f"upscaling from {w}x{h} to {final_width}x{final_height}, scale_factor={scale_factor}")

# run realesrgan video inference
subprocess.run(f"""
python inference_realesrgan_video.py -n {model} -i "{video_path}" -o "{output_dir}" --outscale {scale_factor} --tile 256 --tile_pad 10
""", shell=True)

name = os.path.basename(video_path).replace(".mp4","")
temp_out = os.path.join(output_dir, f"{name}_out.mp4")
final_out = os.path.join(output_dir, f"{name}_upscaled_{final_width}_{final_height}.mp4")

# crop if resolution is not "original"
if "original" not in resolution:
    command = f"""
    ffmpeg -loglevel error -hwaccel cuda -y -i "{temp_out}" \
    -filter:v "crop={final_width}:{final_height}:(in_w-{final_width})/2:(in_h-{final_height})/2" \
    -c:v libx264 -pix_fmt yuv420p "{final_out}"
    """
else:
    command = f"cp '{temp_out}' '{final_out}'"

subprocess.run(command, shell=True)
os.remove(temp_out)
print(f"upscaled video saved to: {final_out}")

# optional: save to drive
if mount_drive:
    drive_dir = "/content/gdrive/MyDrive/Upscaled Videos (REAL-ESRGAN)"
    os.makedirs(drive_dir, exist_ok=True)
    subprocess.run(f"cp '{final_out}' '{drive_dir}/{os.path.basename(final_out)}'", shell=True)
    print(f"saved to drive: {drive_dir}/{os.path.basename(final_out)}")

# display video in colab
from IPython.display import Video, display
display(Video(final_out, embed=True))